# Домашнє завдання 14. Transformers і готові NLP-моделі

Дата заняття: 02.07.2026 (чт)
Модуль: Модуль 6. Обробка природної мови

##### Мета
Навчитися запускати готові transformer-моделі через Hugging Face pipeline, порівнювати якість генерації тексту і критично оцінювати результати простої NLP-класифікації.

#### Завдання
Створити notebook homework_14_transformers.ipynb.

## Завдання 1. Порівняння gpt2, distilgpt2 і tiny-gpt2
Порівняйте 2-3 малі моделі для одного prompt-а.

1. Візьміть один prompt для генерації.
2. Запустіть його на моделях:
    * sshleifer/tiny-gpt2;
    * distilgpt2;
    * gpt2, якщо ПК справляється.
3. Для кожної моделі запишіть:
    * чи швидко вона завантажилась;
    * чи швидко згенерувала текст;
    * наскільки осмисленим був результат;
    * чи були повтори або дивні фрази.
4. Зробіть короткий висновок: яку модель краще брати для швидкої демонстрації, а яку - для трохи якіснішого результату.

## Завдання 2. Простий фільтр токсичних або різких повідомлень
Спробуйте використати sentiment model як дуже простий фільтр ризикових повідомлень.

1. Створіть список із 10 повідомлень користувачів: ввічливі, роздратовані, 2. нейтральні.
2. Запустіть sentiment-analysis.
3. Позначте повідомлення як needs_review, якщо модель повернула NEGATIVE з високим score.
4. Перегляньте результати вручну.
5. Напишіть висновок: чому такий підхід може бути корисним як перший фільтр, але не повинен бути єдиним рішенням у реальному продукті.
Що здати
Файл homework_14_transformers.ipynb.

In [7]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import time 

In [8]:
prompt = "The weather today is"

In [9]:
def test_model(model_name, prompt):

    start_total = time.time()

    start_load = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    load_time = time.time() - start_load
    
    print(f"Time taken to load model: {load_time} seconds")

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    start_generate = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=50, 
        pad_token_id=tokenizer.eos_token_id
    )
    generate_time = time.time() - start_generate

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    total_time = time.time() - start_total
    return {
        'text': text,
        'prompt': prompt,
        'model': model_name,
        'total_time': total_time,
        'generate_time': generate_time
    }


In [10]:
models_to_test = [
    "sshleifer/tiny-gpt2",
    "distilgpt2",
    ]



In [11]:
results = []
for model_name in models_to_test:
    result = test_model(model_name, prompt)
    if result:
        results.append(result)
for result in results:
    print(f"Модель: {result['model']}")
    print(result['text'])
    print(f"Час генерації: {result['generate_time']} секунд")
    print(f"Загальний час: {result['total_time']} секунд")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Time taken to load model: 1.8411245346069336 seconds


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Time taken to load model: 1.7435095310211182 seconds
Модель: sshleifer/tiny-gpt2
The weather today is factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors
Час генерації: 0.16701126098632812 секунд
Загальний час: 2.0081357955932617 секунд
Модель: distilgpt2
The weather today is a bit of a shock, but it's still a good day for the city.



The city of St. Louis is expected to be in the midst of a major storm surge, with the city expected to be in the midst of
Час генерації: 2.583078384399414 секунд
Загальний час: 4.326587915420532 секунд


Обидві моделі завантажуються швидко, але завантаження було повторне, можливо інформація була вже збережена в кеші. На мою думку, це єдине пояснення швидкості.  Первинне завантаження тривало трохи більше 10 хв. 

Tiny-gpt2 генерує текст швидше, ніж distilgpt2. 

Tiny-gpt2  - модель не здатна до генерації адекватної відповіді, має повтори, така відповідь не має сенсу.  Має велику кількість повторів. 

distilgpt2 - генерує текст краще, має якусь логіку, якщо це можна так назвати. Текс більш якісний. Частково осмислений, але до всередені має обрив. В кінці блоку втрачається суть. Має декілька повторів. 

Аалізуючи дві моделі,  для демонстраці, напевно, я оберу першу Tiny-gpt2 для демонстрації, бо вона швидше завантажилась при першому запуску.  Другу - distilgpt2 -  для трохи якіснішого результату.

Загалом, обидві моделі погані. 

In [12]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Python\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Анютка\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [13]:
messages = [
    # ВВІЧЛИВІ (позитивні)
    "Thank you so much for your help! I really appreciate it.",
    "You are doing a great job! Keep up the excellent work.",
    "Thank you for your quick response. It was very helpful.",
    
    # РОЗДРАТОВАНІ (негативні, токсичні)
    "I can't believe you screwed this up again! This is absolutely ridiculous!",
    "Your service is terrible and your staff is incompetent!",
    "I'm so frustrated with your company! You never fix anything!",
    "This is the worst customer service I have ever experienced!",
    
    # НЕЙТРАЛЬНІ
    "I would like to know more about your products.",
    "Please send me the information about your services.",
    "I'm calling to inquire about my order status."
]

print(f"Підготовлено {len(messages)} повідомлень")

Підготовлено 10 повідомлень


In [14]:
import pandas as pd
from tabulate import tabulate


results = [] 

for i, msg in enumerate(messages, 1):
    result = classifier(msg)
    label = result[0]['label']
    score = result[0]['score']
    
    if label == 'POSITIVE':
        category = 'ВВІЧЛИВІ (позитивні)' if score > 0.95 else 'НЕЙТРАЛЬНІ'
        toxicity = "Ні"
    elif label == 'NEGATIVE':
        category = 'РОЗДРАТОВАНІ (негативні)' if score > 0.95 else 'НЕЙТРАЛЬНІ'
        toxicity = "Так"

    results.append({
        'message': msg,
        'result': result,
        'label': label,
        'score': score,
        'category': category,
        'toxicity': toxicity
    })

    df_results = pd.DataFrame(results)
    print(tabulate(df_results, headers='keys', tablefmt='grid', showindex=False))
    

+----------------------------------------------------------+------------------------------------------------------+----------+----------+----------------------+------------+
| message                                                  | result                                               | label    |    score | category             | toxicity   |
+==========================================================+======================================================+==========+==========+======================+============+
| Thank you so much for your help! I really appreciate it. | [{'label': 'POSITIVE', 'score': 0.9998514652252197}] | POSITIVE | 0.999851 | ВВІЧЛИВІ (позитивні) | Ні         |
+----------------------------------------------------------+------------------------------------------------------+----------+----------+----------------------+------------+
+----------------------------------------------------------+------------------------------------------------------+----------+----

In [19]:
for r in results:
    if r['label'] == 'NEGATIVE' and r['score'] > 0.95:
        r['need_review'] = 'Потрібна перевірка'
    else:
        r['need_review'] = "Перевірка не потрібна"

df = pd.DataFrame(results)

print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))

+---------------------------------------------------------------------------+------------------------------------------------------+----------+----------+--------------------------+------------+-----------------------+
| message                                                                   | result                                               | label    |    score | category                 | toxicity   | need_review           |
+===========================================================================+======================================================+==========+==========+==========================+============+=======================+
| Thank you so much for your help! I really appreciate it.                  | [{'label': 'POSITIVE', 'score': 0.9998514652252197}] | POSITIVE | 0.999851 | ВВІЧЛИВІ (позитивні)     | Ні         | Перевірка не потрібна |
+---------------------------------------------------------------------------+-----------------------------------------------

Подібнийй аналіз зручний для обробки великої кількості повідомлень в автоматичному режимі цілодобово. На відміну від людини, не втомлюється. Має чітку умову розподілення на позитив і негатив, не залежить від настрою, віросповідання, упередженого ставлення.  

Але sentiment analysi  не розуміє і немає градуювання терміновості, критичності, не розуміє в повній мірі контекст повідомлення.